# Step 5 V2 — Subspace Analysis (Interpretability)

This notebook produces the **mechanistic understanding** figures for the paper. These explain *why* SPA works, which is critical for NeurIPS acceptance.

## Figures produced

| Figure | Insight |
|---|---|
| 5A: Gradient alignment per round | Projection kills alignment; baseline doesn't |
| 5B: Layer-wise subspace importance | Which layers matter most for safety |
| 5C: Layer-wise ablation | Remove projection from each layer group, measure parity gap |
| 5D: Singular value spectrum | Safety subspace is genuinely low-rank |
| 5E: Hidden state UMAP | Safe vs. unsafe prompts separate after SPA |
| 5F: Subspace projection effect | Energy removed from gradient by projection per round |

## Requirements
- Step 1 outputs (safety subspaces)
- Step 2 outputs (round-level alignment data from main_results.pt)
- Step 4 outputs (group_scores for qualitative analysis)

In [ ]:
import sys
print('=' * 60)
print('ENVIRONMENT')
print('=' * 60)
print(f'Python: {sys.version}')
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} ({p.total_memory/1024**3:.1f} GB)')
print('=' * 60)

In [ ]:
%pip install -q transformers datasets peft matplotlib seaborn umap-learn

In [ ]:
import sys
sys.path.insert(0, '.')
from utils import *
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from peft import PeftModel

GPU_ID  = 0
_DEVICE = torch.device(f'cuda:{GPU_ID}' if torch.cuda.is_available() else 'cpu')
print(f'Device: {_DEVICE}')

In [ ]:
# ── Figure 5A: Gradient alignment per round ───────────────────────────────────
# Load alignment_per_round from Step 2 main results
main_results_path = RESULTS_DIR / 'step2_main_results.pt'

if not main_results_path.exists():
    print('Run Step 2 first.')
else:
    main_results = torch.load(str(main_results_path))
    colors = {'fedavg_noproj': '#e74c3c', 'fedavg_proj': '#2980b9',
              'fedprox': '#f39c12',      'fedprox_proj': '#27ae60'}

    # Aggregate alignment across seeds
    method_alignments = {}
    for r in main_results:
        m = r['method']
        if not r['alignment_per_round']:
            continue
        method_alignments.setdefault(m, []).append(r['alignment_per_round'])

    fig, ax = plt.subplots(figsize=(9, 5))
    for mname, all_align in method_alignments.items():
        max_len = max(len(a) for a in all_align)
        padded = [a + [a[-1]] * (max_len - len(a)) for a in all_align]
        arr = np.array(padded)
        mean = arr.mean(axis=0)
        std  = arr.std(axis=0)
        x = range(1, len(mean) + 1)
        c = colors.get(mname, 'gray')
        ax.plot(x, mean, 'o-', color=c, linewidth=2, markersize=5, label=mname)
        ax.fill_between(x, mean - std, mean + std, alpha=0.15, color=c)

    ax.set_xlabel('FL Round', fontsize=12)
    ax.set_ylabel('Gradient Alignment to Safety Subspace\n'
                  r'$\|U^T g\|^2 / \|g\|^2$', fontsize=11)
    ax.set_title('Figure 5A: Gradient Alignment During Federated Training\n'
                 '(5 seeds, mean ± std)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.4)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.7, alpha=0.5)
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / 'fig5a_gradient_alignment.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: fig5a_gradient_alignment.png')

In [ ]:
# ── Figure 5B: Layer-wise subspace energy (seed=42) ───────────────────────────
subspace_path = MODELS_DIR / 'safety_subspace_seed42.pt'

if not subspace_path.exists():
    print('Run Step 1 first (seed=42 needed).')
else:
    ss = SafetySubspace.load(subspace_path)
    layer_names = list(ss.singular_values.keys())
    k = SAFETY_RANK

    # Energy captured = sum(S[:k]^2) / sum(S^2)
    layer_energies = []
    for lname in layer_names:
        S = ss.singular_values[lname].float()
        energy = (S[:k].pow(2).sum() / S.pow(2).sum().clamp(min=1e-12)).item()
        layer_energies.append(energy)

    fig, ax = plt.subplots(figsize=(14, 4))
    x = np.arange(len(layer_names))
    # Color by module type
    module_colors = []
    for lname in layer_names:
        if 'q_proj' in lname or 'k_proj' in lname or 'v_proj' in lname or 'o_proj' in lname:
            module_colors.append('#3498db')
        else:
            module_colors.append('#e74c3c')

    bars = ax.bar(x, layer_energies, color=module_colors, edgecolor='none', width=1.0)
    ax.axhline(y=np.mean(layer_energies), color='black', linestyle='--',
               linewidth=1.5, label=f'Mean={np.mean(layer_energies):.3f}')
    ax.set_xlabel('Layer Index (ordered by model depth)', fontsize=11)
    ax.set_ylabel(f'Energy captured by top-{k} singular vectors', fontsize=11)
    ax.set_title(f'Figure 5B: Per-Layer Safety Subspace Quality (k={k}, seed=42)',
                 fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1)
    attn_patch = mpatches.Patch(color='#3498db', label='Attention (q/k/v/o)')
    mlp_patch  = mpatches.Patch(color='#e74c3c', label='MLP (gate/up/down)')
    ax.legend(handles=[attn_patch, mlp_patch] + [plt.Line2D([0],[0], color='black',
              linestyle='--', label=f'Mean={np.mean(layer_energies):.3f}')], fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / 'fig5b_layer_energy.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Mean layer energy: {np.mean(layer_energies):.4f}')
    print('Saved: fig5b_layer_energy.png')

In [ ]:
# ── Figure 5C: Layer-group ablation of projection ──────────────────────────────
# Strategy: run FL 5 rounds with projection disabled for {attention-only, MLP-only, all, none}
# This requires a quick FL run — use seed=42, 3 rounds for speed

def run_fl_layer_group(
    safety_bases, rejected_texts, chosen_texts, tokenizer,
    proj_group,   # 'all', 'none', 'attention', 'mlp'
    n_rounds=5, seed=42,
):
    """Run FL with projection limited to a subset of layer types."""
    # Filter safety bases by module type
    if proj_group == 'all':
        active_bases = safety_bases
    elif proj_group == 'none':
        active_bases = {}
    elif proj_group == 'attention':
        active_bases = {k: v for k, v in safety_bases.items()
                        if any(t in k for t in ['q_proj', 'k_proj', 'v_proj', 'o_proj'])}
    elif proj_group == 'mlp':
        active_bases = {k: v for k, v in safety_bases.items()
                        if any(t in k for t in ['gate_proj', 'up_proj', 'down_proj'])}
    else:
        active_bases = safety_bases

    base, _ = load_base_model_and_tokenizer(gpu_id=GPU_ID)
    lora_path = MODELS_DIR / f'safety_lora_seed{seed}'
    if lora_path.exists():
        model = PeftModel.from_pretrained(base, str(lora_path), is_trainable=True)
    else:
        model = apply_lora(base)

    client_ds = prepare_client_datasets_iid(rejected_texts, tokenizer, N_CLIENTS, seed)
    use_proj = (proj_group != 'none')
    _, round_losses, alignment = run_fl_experiment(
        model, tokenizer, client_ds, active_bases,
        n_rounds=n_rounds, use_proj=use_proj, mu=0.0, device=_DEVICE,
        track_alignment=True,
    )
    final_loss = round_losses[-1] if round_losses else float('nan')
    del base, model
    torch.cuda.empty_cache()
    return final_loss, round_losses, alignment


ablation_path = RESULTS_DIR / 'step5_layer_ablation.pt'
if ablation_path.exists():
    ablation_results = torch.load(str(ablation_path))
    print('Loaded existing layer ablation results.')
else:
    ablation_results = {}
    seed = 42
    rejected_texts = torch.load(str(DATA_DIR / f'rejected_texts_seed{seed}.pt'))
    chosen_texts, _ = load_hh_rlhf(n_chosen=N_SAFETY_SAMPLES, n_rejected=0, seed=seed)
    ss = SafetySubspace.load(MODELS_DIR / f'safety_subspace_seed{seed}.pt')
    _, tok = load_base_model_and_tokenizer(gpu_id=GPU_ID)

    for group in ['none', 'attention', 'mlp', 'all']:
        print(f'\nRunning ablation: proj_group={group}')
        final_loss, rl, al = run_fl_layer_group(
            ss.bases, rejected_texts, chosen_texts, tok,
            proj_group=group, n_rounds=5, seed=seed,
        )
        ablation_results[group] = {'final_loss': final_loss, 'round_losses': rl,
                                   'alignment': al}
        torch.save(ablation_results, str(ablation_path))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
group_order = ['none', 'attention', 'mlp', 'all']
group_labels = ['No Proj\n(baseline)', 'Attn only', 'MLP only', 'All layers\n(SPA)']
colors_abl = ['#e74c3c', '#3498db', '#9b59b6', '#27ae60']

final_losses = [ablation_results.get(g, {}).get('final_loss', 0) for g in group_order]
axes[0].bar(group_labels, final_losses, color=colors_abl, edgecolor='black', linewidth=0.8)
axes[0].set_ylabel('Final Round Avg Loss', fontsize=11)
axes[0].set_title('Projection Group Ablation: Final Loss', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.4)

for g, lab, c in zip(group_order, group_labels, colors_abl):
    if g in ablation_results and ablation_results[g]['round_losses']:
        rl = ablation_results[g]['round_losses']
        axes[1].plot(range(1, len(rl)+1), rl, 'o-', label=lab.replace('\n', ' '),
                     color=c, linewidth=2, markersize=5)
axes[1].set_xlabel('FL Round', fontsize=11)
axes[1].set_ylabel('Avg Training Loss', fontsize=11)
axes[1].set_title('Projection Group Ablation: Loss Curves', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.4)

plt.suptitle('Figure 5C: Layer-Group Projection Ablation (seed=42, 5 rounds)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'fig5c_layer_group_ablation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig5c_layer_group_ablation.png')

In [ ]:
# ── Figure 5D: Full singular value spectrum + energy curve ────────────────────
# Show that the safety subspace is genuinely low-rank (motivates SVD approach)
ss = SafetySubspace.load(MODELS_DIR / 'safety_subspace_seed42.pt')

# Aggregate singular value distributions across all layers
all_singular_vals = []
for lname, S in ss.singular_values.items():
    S_np = S.float().numpy()
    S_norm = S_np / (S_np.sum() + 1e-12)  # normalize to sum=1
    all_singular_vals.append(S_norm)

all_sv = np.array(all_singular_vals)  # shape: (n_layers, min_rank)
mean_sv = all_sv.mean(axis=0)
std_sv  = all_sv.std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: mean singular value spectrum
x = np.arange(1, len(mean_sv) + 1)
axes[0].plot(x, mean_sv, 'o-', color='steelblue', linewidth=2, markersize=5)
axes[0].fill_between(x, mean_sv - std_sv, mean_sv + std_sv, alpha=0.2, color='steelblue')
axes[0].axvline(x=SAFETY_RANK, color='red', linestyle='--', linewidth=2,
                label=f'k={SAFETY_RANK} (default)')
axes[0].set_xlabel('Singular value index', fontsize=12)
axes[0].set_ylabel('Normalized singular value (mean ± std over layers)', fontsize=11)
axes[0].set_title('Figure 5D: Safety Subspace Singular Value Spectrum', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.4)

# Right: cumulative energy curve
cumulative_energy = np.cumsum(mean_sv)
axes[1].plot(x, cumulative_energy * 100, 'o-', color='darkorange', linewidth=2, markersize=5)
axes[1].axvline(x=SAFETY_RANK, color='red', linestyle='--', linewidth=2,
                label=f'k={SAFETY_RANK}: {cumulative_energy[SAFETY_RANK-1]*100:.1f}%')
axes[1].axhline(y=80, color='gray', linestyle=':', linewidth=1, label='80% energy')
axes[1].axhline(y=90, color='gray', linestyle='-.', linewidth=1, label='90% energy')
axes[1].set_xlabel('Number of singular vectors (k)', fontsize=12)
axes[1].set_ylabel('Cumulative energy captured (%)', fontsize=12)
axes[1].set_title('Cumulative Energy vs. k', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.4)
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'fig5d_singular_spectrum.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Cumulative energy at k={SAFETY_RANK}: {cumulative_energy[SAFETY_RANK-1]*100:.1f}%')
print('Saved: fig5d_singular_spectrum.png')

In [ ]:
# ── Figure 5E: UMAP of last-layer hidden states (safe vs. unsafe prompts) ─────
try:
    import umap
    HAS_UMAP = True
except ImportError:
    print('umap-learn not installed. Run: pip install umap-learn')
    HAS_UMAP = False

if HAS_UMAP:
    # Sample 100 chosen (safe) and 100 rejected (unsafe) prompts from seed=42
    chosen_texts, rejected_texts = load_hh_rlhf(n_chosen=100, n_rejected=100, seed=42)
    all_texts  = chosen_texts + rejected_texts
    all_labels = ['safe'] * len(chosen_texts) + ['unsafe'] * len(rejected_texts)

    def extract_hidden_states(model, tokenizer, texts, device, layer_idx=-1):
        model.eval()
        hidden_states = []
        for text in texts:
            inputs = tokenizer(text, return_tensors='pt', truncation=True,
                               max_length=256).to(device)
            with torch.no_grad():
                out = model(**inputs, output_hidden_states=True)
            hs = out.hidden_states[layer_idx]  # (1, seq, hidden)
            pooled = hs[0].mean(dim=0).cpu().float().numpy()  # (hidden,)
            hidden_states.append(pooled)
        return np.array(hidden_states)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    label_colors = {'safe': '#27ae60', 'unsafe': '#e74c3c'}

    for ax, (mkey, title) in zip(
        axes,
        [('fedavg_noproj', 'FedAvg (No Proj) — Biased'), ('spa_fedavg', 'SPA (Proj + DPO) — Ours')]
    ):
        seed = 42
        if mkey == 'fedavg_noproj':
            lora_path = MODELS_DIR / f'safety_lora_seed{seed}'
            agg_path  = MODELS_DIR / f'step2_fedavg_noproj_seed{seed}.pt'
        else:
            lora_path = MODELS_DIR / f'step3_fedavg_proj_seed{seed}'
            agg_path  = None

        if not (lora_path.exists() if lora_path else True):
            ax.text(0.5, 0.5, 'Model not found', ha='center', va='center',
                    transform=ax.transAxes, fontsize=12)
            ax.set_title(title)
            continue

        base, tok = load_base_model_and_tokenizer(gpu_id=GPU_ID, for_generation=True)
        if lora_path and lora_path.exists():
            model = PeftModel.from_pretrained(base, str(lora_path), is_trainable=False)
        else:
            model = base

        if agg_path and agg_path.exists():
            agg = torch.load(str(agg_path), map_location='cpu')
            sd = model.state_dict()
            for k_, v in agg.items():
                if k_ in sd:
                    sd[k_] = v.to(dtype=DTYPE)
            model.load_state_dict(sd, strict=False)

        model.eval()
        print(f'Extracting hidden states for {title}...')
        hs = extract_hidden_states(model, tok, all_texts, _DEVICE)

        reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15)
        emb = reducer.fit_transform(hs)

        for label in ['safe', 'unsafe']:
            mask = [l == label for l in all_labels]
            ax.scatter(emb[mask, 0], emb[mask, 1],
                       c=label_colors[label], label=label, s=40, alpha=0.7,
                       edgecolors='black', linewidths=0.3)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.legend(fontsize=10)
        ax.set_xlabel('UMAP-1', fontsize=10)
        ax.set_ylabel('UMAP-2', fontsize=10)

        del base, model
        torch.cuda.empty_cache()

    plt.suptitle('Figure 5E: UMAP of Last-Layer Hidden States (Safe vs. Unsafe Prompts)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / 'fig5e_umap_hidden_states.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: fig5e_umap_hidden_states.png')

In [ ]:
# ── Figure 5F: Energy removed by projection per round ─────────────────────────
# Compute: for each round's gradient, what fraction of energy was projected away?
# Use gradient alignment from main_results (alignment = fraction IN subspace,
# so 'energy removed' = alignment, because projection removes exactly that energy)

main_results = torch.load(str(RESULTS_DIR / 'step2_main_results.pt'))

fig, ax = plt.subplots(figsize=(9, 5))
colors = {'fedavg_noproj': '#e74c3c', 'fedavg_proj': '#2980b9',
          'fedprox': '#f39c12', 'fedprox_proj': '#27ae60'}

for mname in ['fedavg_proj', 'fedprox_proj']:
    all_alignments = [r['alignment_per_round'] for r in main_results
                      if r['method'] == mname and r['alignment_per_round']]
    if not all_alignments:
        continue
    max_len = max(len(a) for a in all_alignments)
    padded  = [a + [a[-1]] * (max_len - len(a)) for a in all_alignments]
    arr     = np.array(padded)
    mean, std = arr.mean(0), arr.std(0)
    x = range(1, len(mean) + 1)
    c = colors[mname]
    ax.plot(x, mean * 100, 'o-', color=c, linewidth=2, markersize=6, label=mname)
    ax.fill_between(x, (mean - std) * 100, (mean + std) * 100, alpha=0.15, color=c)

ax.set_xlabel('FL Round', fontsize=12)
ax.set_ylabel('% Gradient Energy in Safety Subspace\n(= energy removed by projection)', fontsize=11)
ax.set_title('Figure 5F: Safety Gradient Energy Removed Per Round\n'
             '(5 seeds, mean ± std)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'fig5f_projection_energy.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig5f_projection_energy.png')

In [ ]:
# ── Extra: Safety rank k ablation vs. final downstream metrics ────────────────
# If you ran Step 4 with multiple k values, aggregate here.
# Otherwise, use singular value energy as proxy.

RANK_ABLATION_VALUES = [1, 2, 4, 8, 16, 32]
ss = SafetySubspace.load(MODELS_DIR / 'safety_subspace_seed42.pt')

rank_vs_energy = []
for k in RANK_ABLATION_VALUES:
    energies = []
    for lname, S in ss.singular_values.items():
        S = S.float()
        if len(S) >= k:
            captured = (S[:k].pow(2).sum() / S.pow(2).sum().clamp(min=1e-12)).item()
            energies.append(captured)
    rank_vs_energy.append(np.mean(energies))

fig, ax = plt.subplots(figsize=(7, 4))
ax2 = ax.twinx()

ax.plot(RANK_ABLATION_VALUES, rank_vs_energy, 'o-',
        color='steelblue', linewidth=2, markersize=8, label='Subspace energy captured')
ax.axvline(x=SAFETY_RANK, color='red', linestyle='--', linewidth=1.5, label=f'k={SAFETY_RANK} (default)')
ax.set_xlabel('Safety rank k', fontsize=12)
ax.set_ylabel('Mean subspace energy captured', fontsize=11, color='steelblue')
ax.set_ylim(0, 1)
ax.set_xticks(RANK_ABLATION_VALUES)
ax.tick_params(axis='y', labelcolor='steelblue')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

ax.set_title('Safety Rank k vs. Subspace Energy\n(Use with Step 4 parity gap for full ablation)',
             fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'fig_rank_energy_ablation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_rank_energy_ablation.png')

for k, e in zip(RANK_ABLATION_VALUES, rank_vs_energy):
    print(f'  k={k:2d}: energy={e:.4f}')

In [ ]:
import os
plots = sorted(os.listdir(str(PLOTS_DIR)))
print('Step 5 V2 complete.')
print(f'All plots saved to: {PLOTS_DIR}')
for p in plots:
    print(f'  {p}')